In [2]:
from beir.datasets.data_loader import GenericDataLoader
from helpers.converter import convert_corpus_jsonl
import os

/Users/dieudonne/Developer/dense-sparse/.venv/lib/python3.13/site-packages/beir/datasets/data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
## necessary directories
data_path = "./datasets/nfcorpus/" ## For the normal docs
index_path = "datasets/index/" ## For the inverted index
output_doc_path = "./datasets/docs" ## for the converted docs 
qrel_file = "./datasets/nfcorpus/qrels/test.tsv"


In [4]:
## Load the dataset
corpus, queries, qrels = GenericDataLoader(data_path).load()

100%|██████████| 3633/3633 [00:00<00:00, 163192.18it/s]


In [5]:
## Convert the dataset to jsonl
new_data_path = convert_corpus_jsonl(corpus=corpus, output_path=output_doc_path)

Saved 3633 documents to ./datasets/docs/corpus.jsonl


In [6]:
from sparse.bm25 import BM25Retrieval


# Create the index folder if not exist
os.makedirs(index_path, exist_ok=True)

bm25 = BM25Retrieval(index_path) ## We create the index it does not exist
bm25.index(new_data_path) ## Index the data

apr 15, 2026 6:19:13 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFORMAZIONI: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


In [7]:
results = {}

for qid, query in queries.items():
    hits = bm25.searcher.search(query, k=10)
    
    results[qid] = {}
    for hit in hits:
        results[qid][hit.docid] = hit.score
    
    

In [8]:
# run_file = "my_run.trec.txt"

# with open(run_file, "w") as fw:
#     for qid, query in queries.items():
#         hits = bm25.searcher.search(query, k=10)
        
#         for rank, hit in enumerate(hits, 1):
#             fw.write(f"{qid}\tQ0\t{hit.docid}\t{rank}\t{hit.score}\tPyseriniRun\n")


In [9]:
### Evaluate the bm25
from evaluation.metrics import compute_ndcg

metrics = compute_ndcg(qrels, results)

In [ ]:
metrics

AttributeError: 'tuple' object has no attribute 'ndcg'